## 1. 데이터 추출 에이전트의 디코딩 최적화

In [ ]:
import json
import os
from openai import OpenAI

# 1. OpenAI 클라이언트 설정
client = OpenAI(api_key="YOUR_API_KEY")

def get_extraction_config():
    """
    데이터 파싱 에러를 방지하기 위해 모델의 무작위성을 완전히 제거하도록 설정하세요.
    """
    config = {
        # 확률 분포를 뾰족하게 만들어 변동성을 제거 (0.0: Greedy Search와 유사)
        "temperature": 0.0,

        # 확률 상위 0%의 토큰만 고려 (Temperature 0과 함께 사용 시 최상위 1개만 선택)
        "top_p": 0.0,

        # 새로운 토큰 출현에 대한 페널티를 없애 답변의 일관성 유지
        "presence_penalty": 0.0,

        # (참고) OpenAI API에서는 temperature=0 설정 시 내부적으로 최적의 결정론적 경로를 선택합니다.
        # do_sample은 주로 오픈소스 모델(Hugging Face)에서 Greedy Search를 명시할 때 사용합니다.
    }
    return config

# ---------------------------------------------------------
# [테스트 실행] 설정한 Config를 적용하여 실제 답변 생성
# ---------------------------------------------------------

def run_extraction_test(user_input):
    print(f"📥 입력 텍스트: {user_input}")

    # 학생이 설정한 파라미터 로드
    conf = get_extraction_config()

    try:
        # 실제 API 호출
        response = client.chat.completions.create(
            model="gpt-5.4-nano",
            messages=[
                {"role": "system", "content": "너는 JSON 데이터 추출기야. 오직 JSON 형식으로만 답해."},
                {"role": "user", "content": f"다음 문장에서 주문 번호만 추출해: {user_input}"}
            ],
            # 학생이 설정한 config를 언패킹(**)하여 전달
            temperature=conf["temperature"],
            top_p=conf["top_p"],
            presence_penalty=conf["presence_penalty"]
        )

        raw_output = response.choices[0].message.content
        print(f"\n📤 모델 응답(Raw): {raw_output}")

        # JSON 파싱 검증
        parsed_data = json.loads(raw_output)
        print("\n✅ 파싱 성공!")
        print(f"🎯 최종 결과: {parsed_data}")

    except json.JSONDecodeError:
        print("\n❌ 파싱 실패: 응답에 JSON 외의 텍스트가 포함되어 있습니다.")
    except Exception as e:
        print(f"\n⚠️ 기타 오류 발생: {e}")

test_input = "2026년 4월에 주문한 번호 상품 14개 언제 배송되나요? 상품번호는 98765 입니다."
run_extraction_test(test_input)

## 2. Few-shot을 활용한 구조적 데이터 추출 최적화

[시나리오]
당신은 뉴스 기사에서 **'인물'**과 **'소속 기관'**을 찾아 JSON 형식으로 추출하는 에이전트를 개발하고 있습니다. 하지만 Zero-shot(예시 없음)으로 실행했을 때, 모델이 자꾸 불필요한 설명을 덧붙이거나 리스트 형식을 무시하는 실패 케이스가 발생합니다.

실패 케이스 (Zero-shot 결과):

"기사에서 찾은 인물은 홍길동이며, 소속은 활빈당입니다. JSON 형식은 다음과 같습니다: { "name": "홍길동", "org": "활빈당" }"
(문제점: 순수 JSON만 반환해야 하는데 앞뒤 설명이 붙음)

In [ ]:
import openai

def extract_entities_with_few_shot(news_text):
    client = openai.OpenAI()

    # 1. 모델에게 일관된 출력 형식을 학습시키기 위한 Few-shot 예시를 작성하세요.
    FEW_SHOT_EXAMPLES = [

    ]

    # 2. 메인 프롬프트 구성
    messages = [

    ]
    messages.extend(FEW_SHOT_EXAMPLES)
    messages.append(

        )

    response = client.chat.completions.create(
        model="gpt-5.4-nano",
        messages=messages,
        temperature=0.0 # 정확도를 위해 온도 낮춤
    )

    return response.choices[0].message.content

# 테스트 실행
test_news = "이순신 장군이 거북선을 이끌고 조선 수군과 함께 출정했다."
# 예상 결과: [{"person": "이순신", "org": "조선 수군"}]

## 3. LLM의 구조화된 데이터 추출을 위한 프롬프트 엔지니어링

In [ ]:
import json
import os
from openai import OpenAI

# 1. API 키 설정 (본인의 키를 입력하세요)
client = OpenAI(api_key="YOUR_API_KEY")

# [입력 데이터] 비정형 영수증 텍스트
receipt_text = """
2024년 3월 15일 오후 7시 30분, 강남역 인근 '맛있는 파스타'에서 결제함.
주문 메뉴는 알리오올리오 15,000원, 고르곤졸라 피자 22,000원, 콜라 2잔 6,000원임.
총 합계 금액은 43,000원이고 부가세 포함임. 카드 승인번호는 12345678임.
"""

def call_llm_for_extraction(text, error_message=None):
    """
    LLM이 정확한 JSON을 출력하고, 에러 발생 시 스스로 교정할 수 있도록 프롬프트를 설계하세요.
    """

    # 1. LLM에게 역할(Persona)과 출력 규칙을 부여하는 시스템 프롬프트를 작성하세요.
    # (팁: 출력 형식을 JSON으로 제한하고, 설명 없이 결과만 출력하도록 지시하세요.)
    system_prompt = ""

    # 2. 입력 데이터(text)와 에러 메시지(error_message)를 조합하여 사용자 프롬프트를 설계하세요.
    # (팁: error_message가 있을 경우, 모델에게 '어떤 에러'가 났는지 알려주어 수정하게 하세요.)
    if error_message:
        # 재시도 시의 프롬프트
        user_content = f""
    else:
        # 첫 시도 시의 프롬프트
        user_content = f""

    response = client.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content}
        ],
        temperature=0
    )
    return response.choices[0].message.content

# ---------------------------------------------------------
# 아래는 시스템 로직입니다. 수정하지 마세요.
# ---------------------------------------------------------
def safe_extract_receipt(text):
    MAX_RETRIES = 3
    attempt = 0
    error_log = None

    while attempt < MAX_RETRIES:
        attempt += 1
        print(f"\n[시도 {attempt}] 데이터 추출 중...")

        raw_output = call_llm_for_extraction(text, error_log)

        try:
            # 1. JSON 형식 검증
            data = json.loads(raw_output)

            # 2. 필수 필드 검증
            required_fields = ["store_name", "date", "items", "total_amount"]
            for field in required_fields:
                if field not in data:
                    raise KeyError(f"필수 필드 '{field}'가 누락되었습니다.")

            # 3. 데이터 타입 검증
            if not isinstance(data["total_amount"], int):
                raise TypeError("total_amount는 숫자(int)여야 합니다.")

            print("✅ JSON 파싱 및 검증 성공!")
            return data

        except Exception as e:
            error_log = str(e)
            print(f"❌ 오류 발견: {error_log}")

    return None

# [자동 채점 로직]
print("=== 과제 테스트 시작 ===")
final_result = safe_extract_receipt(receipt_text)

if final_result:
    score = 0
    if final_result.get("store_name") == "맛있는 파스타": score += 25
    if final_result.get("total_amount") == 43000: score += 25
    if len(final_result.get("items", [])) == 3: score += 25
    if final_result.get("date") == "2024-03-15": score += 25

    print("\n" + "="*30)
    print(f"최종 채점 점수: {score}/100")
    print(f"최종 데이터: {final_result}")
    print("="*30)
else:
    print("\n정답 추출 실패: 모든 재시도에서 유효한 JSON을 얻지 못했습니다.")

## 4. 프롬프트 엔지니어링 e2e 작성

다음과 같은 자연어 입력이 주어진다:

"나는 어제 친구랑 영화를 봤는데 너무 재미없었어"

이 문장을 아래 JSON 포맷으로 변환해야 한다:

{
  "sentiment": "negative",
  "reason": "..."
}

[문제조건] 1. 반드시 JSON만 출력해야 함
2. 항상 valid JSON이어야 함
3. sentiment는 이 중 하나만 허용
"positive"
"negative"
"neutral"
4.

In [ ]:
from openai import OpenAI
import json

client = OpenAI()

# 1. 3개의 입력을 전부 올바르게 추론 하도록 작성
# inputs = [
#     "오늘 날씨가 너무 좋아서 기분이 최고야", #positice
#     "어젠 날씨가 최악이었는데 오늘은 그래도 비가 안 와서 나쁘지 않네", #neutral
#     "지금까지 너무 잘 쓰고 있었는데.. 오늘은 서비스가 최악이었고 다시는 안 쓸거야" #negative
# ]

def run_prompt(prompt, text):
    response = client.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": text}
        ],
        temperature=0.7
    )
    return response.choices[0].message.content

prompt = "..."

text = "나는 어제 친구랑 영화를 봤는데 너무 재미없었어"
output = run_prompt(prompt, text)

print(output)

# JSON validation
try:
    parsed = json.loads(output)
    print("✅ valid JSON")
except:
    print("❌ invalid JSON")

## 5. Chain-of-Thought를 이용한 복잡한 논리 추론 성능 비교

In [ ]:
import os
from openai import OpenAI

# 1. 환경 설정 (본인의 API 키를 입력하세요)
client = OpenAI(api_key="YOUR_API_KEY")

# 문제 정의: 재고 관리 퍼즐
problem_description = """
[재고 관리 퍼즐]
1. 월요일 아침, 창고에는 100개의 노트북이 있었습니다.
2. 오전 중에 20개가 출고되었고, 오후에 불량으로 5개가 반품되었습니다.
3. 화요일에는 전날 남은 재고의 10%를 직원 복지로 증정했습니다. (소수점 발생 시 내림 처리)
4. 수요일에는 새로운 모델이 50개 입고되었으나, 기존 모델 15개가 단종되어 폐기되었습니다.
최종적으로 현재 창고에 있는 노트북은 총 몇 개인가요? 숫자만 답하지 말고 최종 결과값을 명확히 포함해 주세요.
"""

def get_response(prompt):
    """LLM 호출을 위한 기본 함수입니다."""
    response = client.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content

# ---------------------------------------------------------
# [Task 1] Standard Prompt 작성 (Zero-shot)
# 지시사항: 추가 가이드 없이 모델이 바로 문제를 풀도록 프롬프트를 작성하세요.
# ---------------------------------------------------------
# TODO: 아래 변수에 일반적인 질문 형태의 프롬프트를 작성하세요.
standard_prompt = ""

# ---------------------------------------------------------
# [Task 2] Chain-of-Thought (CoT) Prompt 작성
# 지시사항: 모델이 논리적 단계를 밟아 사고하도록 유도하는 프롬프트를 작성하세요.
# (팁: 단계별 사고 유도 문구를 넣거나, 추론 형식을 지정해 보세요.)
# ---------------------------------------------------------
# TODO: 아래 변수에 CoT 기법이 적용된 프롬프트를 작성하세요.
cot_prompt = ""

# 실행 및 결과 출력
print("=== [Task 1: Standard Prompt 실행 결과] ===")
if standard_prompt:
    result_std = get_response(standard_prompt)
    print(result_std)
else:
    print("Standard Prompt가 작성되지 않았습니다.")

print("\n=== [Task 2: CoT Prompt 실행 결과] ===")
if cot_prompt:
    result_cot = get_response(cot_prompt)
    print(result_cot)
else:
    print("CoT Prompt가 작성되지 않았습니다.")

# ---------------------------------------------------------
# [Task 3] 자동 채점 및 결과 분석
# ---------------------------------------------------------
def verify_logic(response_text):
    # 정답: 112개 (월: 85, 화: 77, 수: 112)
    return "112" in response_text

print("\n=== [채점 결과] ===")
if cot_prompt:
    is_correct = verify_logic(result_cot)
    print(f"정답 여부 (CoT): {'✅ Pass' if is_correct else '❌ Fail'}")

질문 1: Standard Prompt 결과에서 모델이 범한 가장 큰 논리적 오류는 무엇인가요? (예: 반품을 출고로 착각, 내림 계산 실수 등)

질문 2: 본인이 작성한 CoT Prompt에서 모델의 추론을 가이드하기 위해 사용한 구체적인 전략은 무엇이며, 그것이 왜 효과적이었다고 생각하나요?

질문 3: 만약 CoT를 썼음에도 오답이 나온다면, 프롬프트를 어떻게 수정하면 더 견고한(Robust) 추론을 이끌어낼 수 있을까요?

## 6. 생성 지표(ROUGE, BLEU)를 활용한 모델 평가

In [ ]:
from torchmetrics.text.rouge import ROUGEScore
from torchmetrics.text.bleu import BLEUScore

# 1. 평가를 위한 데이터 설정
target = ["인공지능 에이전트는 효율적인 업무를 돕는다."]
preds = ["인공지능 에이전트는 효율적인 업무를 지원한다."]

# 2. ROUGE 지표 계산
# ROUGE-L은 가장 긴 공통 부분 수열(LCS)을 기반으로 하며, 주로 '이것'을 측정하는 데 집중합니다.
rouge = ROUGEScore()
rouge_results = rouge(preds, target)
print(f"ROUGE-L Score: {rouge_results['_______']}")

# 3. BLEU 지표 계산
# BLEU는 예측된 결과 안에 정답 단어가 얼마나 포함되었는지인 '이것'을 측정합니다.
bleu = BLEUScore()
bleu_score = bleu(preds, target)
print(f"BLEU Score: {bleu_score}")

4. 위 두 문장에서 '돕는다'와 '지원한다'처럼 의미는 같지만 단어가 다를 경우,
n-gram 기반인 ROUGE와 BLEU 점수는 어떻게 변할까요?

5. 이러한 한계를 극복하기 위해 강의 자료에서 제안하는 최신 평가 방식은?

## 7. 통계적 vs 의미론적 평가 지표 비교 구현

[시나리오]
당신은 한국어 요약 모델의 성능을 정교하게 측정하려 합니다.
단순히 단어가 겹치는 정도를 보는 ROUGE-L(통계적)과 문맥적 의미를 파악하는 BERTScore(의미론적)를 동시에 계산하여, 두 지표의 차이점을 분석하는 파이프라인을 구축하세요.

정답(Reference): "인공지능 에이전트는 사용자의 업무 효율을 높여준다."

예측(Prediction): "AI 비서는 유저의 작업 생산성을 향상시킨다."

분석 포인트: 두 문장은 의미는 거의 같지만, 사용된 단어(토큰)는 거의 겹치지 않습니다.

In [ ]:
# !pip install rouge-score bert-score

from rouge_score import rouge_scorer
from bert_score import score

def calculate_metrics(reference, prediction):
    # 1. ROUGE-L (통계적/단어 중첩 기반) 계산

    # 2. BERTScore (의미론적/컨텍스트 기반) 계산

    return rouge_l_score, bert_f1_score

# 테스트 문장
reference = "인공지능 에이전트는 사용자의 업무 효율을 높여준다."
prediction = "AI 비서는 유저의 작업 생산성을 향상시킨다."

# 실행
rouge_val, bert_val = calculate_metrics(reference, prediction)

print(f"--- 평가 결과 ---")
print(f"Reference: {reference}")
print(f"Prediction: {prediction}")
print(f"1. ROUGE-L F1 Score: {rouge_val:.4f}")
print(f"2. BERTScore F1 Score: {bert_val:.4f}")

try:
    # 1. ROUGE-L은 단어 불일치로 인해 낮은 점수를 기록해야 함
    assert rouge_val < 0.2, "ROUGE-L 점수가 예상보다 높습니다. 단어 일치 기반 알고리즘이 맞는지 확인하세요."

    # 2. BERTScore는 의미적 유사성을 파악하여 상대적으로 높은 점수를 기록해야 함 (Bert 모델에 따라 점수가 다를 수 있음)
    assert bert_val > 0.8, "BERTScore가 너무 낮습니다. 의미론적 유사도를 제대로 파악하고 있는지 확인하세요."

    # 3. 두 지표의 특성 차이 검증
    assert bert_val > rouge_val, "의미론적 지표가 통계적 지표보다 높게 산출되어야 합니다."

    print("\n✅ 모든 테스트 케이스를 통과했습니다! (성능 측정 로직 정상)")

except AssertionError as e:
    print(f"\n❌ 검증 실패: {e}")

## 8. PyTorch를 활용한 Contrastive Decoding 구현

[시나리오]
당신은 LLM의 문장 생성 품질을 개선하기 위해, 고성능 모델(Expert)과 저성능 모델(Amateur)의 출력을 결합하는 ContrastiveSearch 모듈을 개발해야 합니다.

소형 모델이 가지는 반복적이고 뻔한 패턴을 억제하고, 고성능 모델의 지식에 기반한 답변을 유도하는 로직을 작성하세요.

[요구사항 및 구현 조건] Adaptive Masking 구현: Expert 모델의 로짓 중 상위권이 아닌 값들은 노이즈로 간주하여 필터링해야 합니다.
Contrastive Logits 계산: Expert 로짓에서 가중치($\tau$)가 적용된 Amateur 로짓을 감산합니다.Softmax 및 샘플링: 최종 로짓에 소프트맥스를 적용하고, argmax를 통해 최적의 토큰을 선택합니다.외부 라이브러리 미사용: torch와 torch.nn.functional만 사용하여 순수 로직으로 작성하세요.

In [ ]:
import torch
import torch.nn.functional as F

class ContrastiveGenerator:
    def __init__(self, alpha=0.1, tau=1.5):
        """
        alpha: Expert 모델의 신뢰 구간 (Masking 기준)
        tau: Small 모델의 로짓을 감산할 가중치
        """
        self.alpha = alpha
        self.tau = tau

    def get_next_token(self, expert_logits, small_logits):
        """
        Args:
            expert_logits (torch.Tensor): 고성능 대형 모델의 로짓 [Vocab_Size]
            small_logits (torch.Tensor): 저성능 소형 모델의 로짓 [Vocab_Size]

        Returns:
            int: 최종 결정된 다음 토큰 인덱스
        """
        # 아래에 전체 로직을 직접 작성하세요.

        pass

def run_test():
    # 테스트 데이터
    expert_logits = torch.tensor([12.0, 15.0, 4.0])
    small_logits = torch.tensor([10.0, 14.5, 2.0])
    generator = ContrastiveGenerator(alpha=0.1, tau=1.2)

    # 함수 실행
    result = generator.get_next_token(expert_logits, small_logits)

    # 검증: Index 1의 점수가 깎여서 Index 0이 선택되어야 함
    expected_index = 0
    if result == expected_index:
        print(f"✅ 테스트 합격! 선택된 토큰: {result}")
    else:
        print(f"❌ 테스트 실패. 선택된 토큰: {result} (예상: {expected_index})")
        print("Tip: Small 모델의 로짓을 감산하는 로직이 정확한지 확인하세요.")

run_test()

## 9. DSPy 설계 및 구현

1. DSPy Signature 작성하기
사용자의 입력(Prompt)에 악의적인 의도(Prompt Injection 등)가 포함되어 있는지 판별하는 DSPy Signature를 작성해야합니다.
주어진 뼈대 코드를 바탕으로 InputField와 OutputField를 정의하고, 모델이 역할을 잘 수행할 수 있도록 적절한 설명(Description)을 추가해 보세요.

In [ ]:
import dspy

class GuardrailSignature(dspy.Signature):
    """주어진 사용자 입력이 안전한지, 혹은 악의적인 프롬프트 인젝션이 포함되어 있는지 판별합니다."""

    # 빈칸을 채워 코드를 완성하세요.
    user_input = # 1. 입력 필드 정의 (설명 포함)
    is_safe = # 2. 출력 필드 정의 (설명 포함)

2. Chain of Thought (CoT)를 활용한 Module 구현
문제 1에서 만든 GuardrailSignature를 사용하여, LLM이 단순히 결과만 내뱉는 것이 아니라 단계별로 추론(Step-by-step reasoning)하도록 만드는 모듈을 구현하려고 합니다. dspy.ChainOfThought를 활용해 코드를 완성하세요.

In [ ]:
class GuardrailAgent(dspy.Module):
    def __init__(self):
        super().__init__()
        # 빈칸을 채워 코드를 완성하세요.
        # 1. GuardrailSignature에 Chain of Thought를 적용하여 초기화하세요.
        self.analyze_safety =

    def forward(self, user_input):
        # 2. 초기화한 모듈을 호출하여 결과를 반환하세요.
        result =
        return result

3. Hallucination 완화를 위한 Decoding 설정
가드레일 에이전트가 추론 과정을 생성할 때 허위 정보(Hallucination)를 만들어내거나, 같은 단어를 무의미하게 반복하지 않도록 LLM API의 파라미터를 설정하려고 합니다.
강의 자료에 언급된 Nucleus Sampling(Top-p) 방식과 Repetition Penalty를 적용하여 설정 딕셔너리를 완성해 보세요.

In [ ]:
llM_generation_config = {
    "temperature": 0.7,
    # 빈칸을 채워 코드를 완성하세요.
    # 1. 누적 확률 분포가 95%가 되는 토큰 집합 내에서 샘플링하도록 설정
    "_______": _______,

    # 2. 동일한 토큰이 반복해서 생성되는 것을 방지하기 위한 패널티 설정 (기본값 1.0보다 높게 설정, 예: 1.2)
    "_______": _______
}

## 10. Self-Correction 프롬프트 개선 루프 구현

[시나리오] 과제 설명:
사용자의 요구사항은 때로 매우 까다롭고 상충할 때가 있습니다. LLM이 한 번에 해결하기 어려운 문제를 **[생성 -> 비판 -> 프롬프트 개정 -> 재생성]**의 3단계 자동 개선 루프를 통해 해결하는 에이전트를 설계하세요.

In [ ]:
import os
import re
from openai import OpenAI

# API 설정 (수강생 본인의 API 키 입력)
client = OpenAI()

# LLM이 한 번에 지키기 까다로운 4가지 제약 조건이 포함되어 있습니다.
user_goal = """
신제품 '에코 버즈(무선 이어폰)'의 인스타그램 홍보글을 작성해줘.
다음 4가지 조건을 반드시 모두 지켜야 해:
1. 전체 길이는 공백 포함 '정확히 120자 이상, 150자 이하'여야 해.
2. '친환경', '노이즈캔슬링', '할인'이라는 3개의 단어가 반드시 1번 이상 포함되어야 해.
3. '플라스틱', '최고', '애플'이라는 단어는 절대 사용하면 안 돼.
4. 글의 맨 마지막에는 정확히 3개의 해시태그(#)만 연속해서 작성해 (예: #에코버즈 #친환경 #이벤트). 그 외의 위치에 해시태그를 쓰면 안 돼.
"""

def get_llm_response(system_prompt, user_prompt):
    """LLM API를 호출하는 기본 헬퍼 함수입니다."""
    response = client.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.2
    )
    return response.choices[0].message.content

# ---------------------------------------------------------
# [Task] Iterative Optimization Loop (자동 개선 루프) 구현
# ---------------------------------------------------------

def optimize_prompt_loop(goal):
    # 초기 프롬프트는 사용자의 목표를 그대로 전달합니다.
    current_prompt = f"다음 지시사항을 엄격하게 지켜서 글을 작성해줘:\n{goal}"
    best_answer = ""

    print("프롬프트 자동 최적화 루프를 시작합니다...\n")

    for i in range(10):  # 최대 10회 루프 반복
        print(f"================ [ {i+1}차 시도 ] ================")

        # 1. [생성 단계] 현재 프롬프트로 결과물 생성
        current_answer = get_llm_response("당신은 마케팅 카피라이터입니다.", current_prompt)
        print(f"📝 생성된 결과물 (글자수: {len(current_answer)}자):\n{current_answer}\n")

        # 2. [비판 단계]
        # TODO 1: 모델이 스스로 만든 답변(current_answer)이 목표(goal)의 4가지 조건을 모두 지켰는지 냉정하고 꼼꼼하게 평가하도록 프롬프트를 작성하세요.
        critic_prompt = f"""
        여기에 비판을 수행하는 프롬프트를 작성하세요.
        목표: {goal}
        생성된 글: {current_answer}
        """
        feedback = get_llm_response("당신은 아주 깐깐하고 팩트 기반으로 평가하는 편집장입니다.", critic_prompt)
        print(f"🔍 피드백(비판):\n{feedback}\n")

        # 3. [프롬프트 개정 단계]
        # TODO 2: 평가자의 피드백(feedback)을 바탕으로, 모델이 다음 시도에서 절대 실수하지 않도록 '원래 프롬프트(goal)'를 더 강력하고 구체적인 지시문으로 업그레이드하세요.
        improvement_instruction = f"""
        여기에 프롬프트를 개선하는 지시문을 작성하세요.
        원래 지시사항: {goal}
        편집장 피드백: {feedback}
        """
        current_prompt = get_llm_response("당신은 프롬프트 엔지니어링 전문가입니다.", improvement_instruction)

        best_answer = current_answer

    return best_answer

# ---------------------------------------------------------
# [자동 채점 시스템] - 이 테스트에서 100점을 받아야 합니다.
# ---------------------------------------------------------
def evaluate_final_output(text):
    score = 0
    print("\n📊 [최종 결과물 채점]")

    # 1. 글자 수 검증 (120 ~ 150자)
    length = len(text)
    if 120 <= length <= 150:
        print("✅ 글자 수 통과")
        score += 25
    else:
        print(f"❌ 글자 수 실패 (현재 {length}자)")

    # 2. 필수 단어 검증
    keywords = ["친환경", "노이즈캔슬링", "할인"]
    if all(word in text for word in keywords):
        print("✅ 필수 단어 포함 통과")
        score += 25
    else:
        print("❌ 필수 단어 누락 감지")

    # 3. 금지 단어 검증
    forbidden = ["플라스틱", "최고", "애플"]
    if not any(word in text for word in forbidden):
        print("✅ 금지 단어 미사용 통과")
        score += 25
    else:
        print("❌ 금지 단어 사용 감지")

    # 4. 해시태그 검증 (마지막에 정확히 3개)
    tags = re.findall(r'#\w+', text)
    if len(tags) == 3 and text.strip().endswith(tags[-1]):
        print("✅ 해시태그 조건 통과")
        score += 25
    else:
        print(f"❌ 해시태그 조건 실패 (현재 {len(tags)}개 이거나 맨 끝이 아님)")

    return score

# 메인 실행
final_text = optimize_prompt_loop(user_goal)
final_score = evaluate_final_output(final_text)

print("\n" + "*"*50)
print(f"🏆 최종 채점 점수: {final_score} / 100")
print("*"*50)